In [2]:
import psycopg2
import pandas as pd

In [3]:
# Add family size information to the DataFrame, pull this from the database
DB = {
    "host":     "petadex.ccz9y6yshbls.us-east-1.rds.amazonaws.com",
    "port":     5432,
    "database": "petadex",
    "user":     "readonly_user",
    "password": "petadex",
}

conn = psycopg2.connect(**DB)
cur  = conn.cursor()

In [4]:
cur.execute('SELECT * FROM "60pid_family_clusters";')
rows = cur.fetchall()
df = pd.DataFrame(rows, columns=[desc[0] for desc in cur.description])
num_clusters = max(df["60pid_family_id"])

In [5]:
df

,60pid_family_id,centroid_orf_id,date_clustered
0,1,9,2026-05-25
1,2,17,2026-05-25
2,3,20,2026-05-25
3,4,32,2026-05-25
4,5,35,2026-05-25
...,...,...,...
1814291,1814292,1373160196,2026-05-25
1814292,1814293,1373162328,2026-05-25
1814293,1814294,1373169904,2026-05-25
1814294,1814295,1373172032,2026-05-25


In [39]:
def fetch_sequences(fasta_path, target_ids):
    target_ids = set(str(x) for x in target_ids)
    remaining = set(target_ids)
    sequences = {}
    with open(fasta_path) as f:
        orf_id = None
        seq_parts = []
        for line in f:
            if not remaining:
                break
            line = line.strip()
            if line.startswith('>'):
                if orf_id and orf_id in target_ids:
                    sequences[orf_id] = "".join(seq_parts)
                    remaining.discard(orf_id)
                orf_id = line[1:].split('|')[0]
                seq_parts = []
            elif orf_id in target_ids:
                seq_parts.append(line)
        if orf_id and orf_id in target_ids:
            sequences[orf_id] = "".join(seq_parts)
    return sequences

sequences = fetch_sequences("./data/petadex.catalytic_orfs.v1.1.fa", df["centroid_orf_id"].tolist())
df["sequence"] = df["centroid_orf_id"].map(lambda x: sequences.get(str(x)))
print(df[["centroid_orf_id", "sequence"]].head())

   centroid_orf_id                                           sequence
0                9  MSNGYSACPAPYDHRGMRKHASLLAALPVLLAACGTTTSTLTTPAV...
1               17  MELNSMQFLLGLIGLLLLIVTSLRRWLLRRESPQKQAVDFHGELYQ...
2               20  MMNVLTKCKLALGIIAIFFSLPSFAVPCSDCSNGFERGQVPRVDQL...
3               32  MDEGACSVLRSLSLRIAAVAAAVTLPLFGLVAPAEAESPYERGPDP...
4               35  MPTEVVMPKWGLSMQEGKINLWLKREGEAVQKGEPIAEVETEKITN...


In [40]:
df.to_csv("./data/60pid_family_clusters_with_sequences.csv", index=False)
ids = df["centroid_orf_id"].tolist()

In [41]:
cur.execute('SELECT * FROM "petadex_clustering" WHERE "orf_id" = ANY(%s);', (ids,))

In [42]:
rows = cur.fetchall()
df_clustering = pd.DataFrame(rows, columns=[desc[0] for desc in cur.description])
df_clustering.to_csv("./data/60pid_centroids.csv", index=False)

In [44]:
df_clustering["60pid_family_id"].value_counts()

60pid_family_id
1          1
2          1
3          1
4          1
5          1
          ..
1814292    1
1814293    1
1814294    1
1814295    1
1814296    1
Name: count, Length: 1814296, dtype: int64